# 08 — Variable-road starting advantage
Reuse the completed 5,000-game experiment. The minimum number of additional roads is determined on the frozen initial board by multi-source BFS, respecting opponent roads, settlements, distance rule and road supply. One-road scores are reused; only positions requiring more roads are recalculated. All outputs are separate. No new simulations needed.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from variable_road_model import run, affordable, readiness_curve
DATA=Path("processed_data_5000")
OUT=Path("processed_data_variable_5000")
CHARTS=Path("charts/variable_road_5000")
CHARTS.mkdir(parents=True,exist_ok=True)
# To run with multiprocessing on Windows, use the terminal command in VARIABLE_ROAD_ANALYSIS.md instead.
# If already generated by the CLI, load the result below rather than recomputing.
d=run(DATA,OUT,workers=1)
print("Games:",d.Game_ID.nunique(),"Players:",len(d))
assert d.Game_ID.nunique()==5000 and len(d)==20000

In [ ]:
# Fixed pilot boundaries for a direct comparison.
old_class=np.select([d.One_Road_Blocked,d.One_Road_Turns_To_50.le(7),d.One_Road_Turns_To_50.le(14)],["Blocked","Strong","Average"],default="Weak")
classes=["Strong","Average","Weak","Blocked","No legal path"]
summary=pd.DataFrame({"One road":pd.Series(old_class).value_counts(),"Variable roads":d.Advantage_Class.value_counts()}).reindex(classes).fillna(0).astype(int)
display(summary)
display(d.Minimum_Roads_Required.value_counts(dropna=False))
display(d.groupby("Minimum_Roads_Required")[["Turns_To_50","Prob_Turn_10"]].agg(["count","mean","median"]))
summary.to_csv(OUT/"class_comparison.csv")
ax=summary.plot.bar(rot=0,title="Starting advantage: original vs variable roads")
ax.set_ylabel("Players")
ax.figure.tight_layout(); ax.figure.savefig(CHARTS/"class_comparison.png",dpi=160); plt.show()
ax=d.Minimum_Roads_Required.fillna(0).astype(int).value_counts().sort_index().rename(index={0:"No path"}).plot.bar(rot=0,title="Minimum additional roads")
ax.set_ylabel("Players")
ax.figure.tight_layout(); ax.figure.savefig(CHARTS/"minimum_roads.png",dpi=160); plt.show()

In [ ]:
turns=[1,2,3,5,10,20,40]
curves=pd.DataFrame({"One road":[d[f"One_Road_Prob_Turn_{t}"].mean() for t in turns],"Variable roads":[d[f"Prob_Turn_{t}"].mean() for t in turns]},index=turns)
curves.index.name="Personal_Turn"
curves.to_csv(OUT/"readiness_comparison.csv")
ax=(curves*100).plot(marker="o",title="Readiness by personal turn")
ax.set(xlabel="Personal turn",ylabel="Mean readiness (%)",ylim=(0,100))
ax.figure.tight_layout(); ax.figure.savefig(CHARTS/"readiness_comparison.png",dpi=160); plt.show()
display(pd.crosstab(pd.Series(old_class,name="Original"),d.Advantage_Class))

## Actual outcomes
Use the outcome CSV from notebook 07. If it is missing, run the extraction script once. No post-setup variable is a predictor.

In [ ]:
from outcome_extraction import extract
path=DATA/"actual_outcomes.csv"
if not path.exists():
    extract(Path('data_5000')).to_csv(path,index=False)
o=pd.read_csv(path)
joined=d.merge(o,on=["Game_ID","Player","Player_Number"],validate="one_to_one")
assert len(joined)==len(d)==len(o)
joined["Built_Third"]=joined.Actual_Turns_To_Third.notna()
completed=joined[joined.Completed_Game.eq(1)].copy()
assert completed.groupby("Game_ID").Won.sum().eq(1).all()
print("Completed games:",completed.Game_ID.nunique())
s=completed.groupby("Advantage_Class").agg(Players=("Won","size"),Wins=("Won","sum"),Win_Rate=("Won","mean"),Third_Rate=("Built_Third","mean"))
display(s)
s.to_csv(OUT/"outcome_by_advantage.csv")
joined.to_csv(OUT/"initial_vs_outcome.csv",index=False)
ax=(s.Win_Rate.reindex(["Strong","Average","Weak","No legal path"]).dropna()*100).plot.bar(rot=0,title="Observed win rate by new advantage class")
ax.set_ylabel("Win rate (%)")
ax.figure.tight_layout(); ax.figure.savefig(CHARTS/"win_by_advantage.png",dpi=160); plt.show()
for name,col in [("One road","One_Road_Turns_To_50"),("Variable roads","Turns_To_50")]:
    paired=joined.dropna(subset=[col,"Actual_Turns_To_Third"])
    print(name,len(paired),paired[col].corr(paired.Actual_Turns_To_Third))

In [ ]:
# Identical five game-grouped folds and logistic regression settings for both scores.
import outcome_prediction as op
completed["Road_Count_Feature"]=completed.Minimum_Roads_Required.fillna(14)
original=op.FEATURE_SETS
try:
    op.FEATURE_SETS={"Initial features":op.BASE,
        "Initial + one-road readiness":op.BASE+["One_Road_Prob_Turn_10"],
        "Initial + variable-road readiness":op.BASE+["Prob_Turn_10"],
        "Initial + variable readiness + road count":op.BASE+["Prob_Turn_10","Road_Count_Feature"]}
    metrics,pred=op.compare(completed)
finally:
    op.FEATURE_SETS=original
display(metrics)
metrics.to_csv(OUT/"outcome_prediction_results.csv",index=False)
pred.to_csv(OUT/"outcome_predictions.csv",index=False)
ax=metrics.set_index("Model")[["ROC_AUC","Within_Game_Top1"]].plot.barh(title="Winner prediction comparison")
ax.set_xlim(0,1)
ax.figure.tight_layout(); ax.figure.savefig(CHARTS/"winner_prediction.png",dpi=160); plt.show()

The pilot thresholds (7 and 14) are fixed for comparison, not newly fitted quartiles. 'No legal path' is a technical status retained only when no path exists within remaining road supply. Finite readiness is not observed construction; the timing correlation is conditional on both measurements. The one-road baseline probabilities and original files are preserved.